In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("../../data/rolling_feature_engineered_tier_1_games.csv", sep=",")
df.head()

,previous_10_game_team1_player1_average_kills,previous_10_game_team1_player1_average_deaths,previous_10_game_team1_player1_average_assists,previous_10_game_team1_player1_average_adr,previous_10_game_team1_player1_average_kast,previous_10_game_team1_player1_average_kddiff,previous_10_game_team1_player2_average_kills,previous_10_game_team1_player2_average_deaths,previous_10_game_team1_player2_average_assists,previous_10_game_team1_player2_average_adr,...,team1_player1_id,team1_player2_id,team1_player3_id,team1_player4_id,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id
0,16.000000,15.000000,4.000000,75.000000,70.000000,1.000000,16.000000,15.000000,4.000000,75.000000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
1,12.000000,17.000000,4.000000,59.100000,62.500000,-5.000000,13.000000,11.000000,2.000000,52.600000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
2,13.500000,16.500000,4.500000,66.250000,71.750000,-3.000000,18.500000,11.000000,4.000000,82.550000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
3,14.612903,14.645161,4.258065,69.832258,71.535484,-0.032258,14.612903,14.645161,4.258065,69.832258,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0
4,13.000000,5.000000,1.000000,81.300000,86.700000,8.000000,11.000000,6.000000,4.000000,72.100000,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0


In [3]:
target = "team1_win"

df[target].value_counts(normalize=True)

team1_win
1    0.548836
0    0.451164
Name: proportion, dtype: float64

sort time

In [4]:
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
print(df["datetime"].head())
print(df["datetime"].tail())

0   2023-10-27 11:00:00
1   2023-10-27 11:00:00
2   2023-10-27 11:00:00
3   2023-10-27 14:15:00
4   2023-10-27 14:15:00
Name: datetime, dtype: datetime64[ns]
5534   2026-03-28 19:40:00
5535   2026-03-28 19:40:00
5536   2026-03-28 19:40:00
5537   2026-03-29 13:30:00
5538   2026-03-29 13:30:00
Name: datetime, dtype: datetime64[ns]


Transform player-level data into team-level data.

In [5]:

stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

for stat in stats:
    team1_cols = [
        f"previous_10_game_team1_player{i}_average_{stat}"
        for i in range(1, 6)
    ]
    
    team2_cols = [
        f"previous_10_game_team2_player{i}_average_{stat}"
        for i in range(1, 6)
    ]
    
    df[f"team1_avg_{stat}"] = df[team1_cols].mean(axis=1)
    df[f"team2_avg_{stat}"] = df[team2_cols].mean(axis=1)
    
    df[f"{stat}_diff"] = df[f"team1_avg_{stat}"] - df[f"team2_avg_{stat}"]

# Check new features
new_features = []

for stat in stats:
    new_features.extend([
        f"team1_avg_{stat}",
        f"team2_avg_{stat}",
        f"{stat}_diff"
    ])

df[new_features].head()

,team1_avg_kills,team2_avg_kills,kills_diff,team1_avg_deaths,team2_avg_deaths,deaths_diff,team1_avg_assists,team2_avg_assists,assists_diff,team1_avg_adr,team2_avg_adr,adr_diff,team1_avg_kast,team2_avg_kast,kast_diff,team1_avg_kddiff,team2_avg_kddiff,kddiff_diff
0,16.000000,16.000000,0.0,15.000000,15.000000,0.0,4.000000,4.000000,0.0,75.000000,75.000000,0.00,70.000000,70.000000,0.00,1.000000,1.000000,0.0
1,15.600000,15.000000,0.6,15.000000,15.600000,-0.6,2.800000,6.000000,-3.2,68.160000,67.640000,0.52,68.320000,66.640000,1.68,0.600000,-0.600000,1.2
2,15.100000,13.600000,1.5,13.600000,15.200000,-1.6,3.800000,4.600000,-0.8,70.120000,67.010000,3.11,72.260000,63.790000,8.47,1.500000,-1.600000,3.1
3,14.612903,14.612903,0.0,14.645161,14.645161,0.0,4.258065,4.258065,0.0,69.832258,69.832258,0.00,71.535484,71.535484,0.00,-0.032258,-0.032258,0.0
4,13.600000,4.600000,9.0,4.600000,13.600000,-9.0,4.200000,1.200000,3.0,92.420000,45.620000,46.80,88.000000,40.000000,48.00,9.000000,-9.000000,18.0


The model compares the difference in average scores between team1 and team2 on the map in the past.

In [6]:

df["map_score_diff"] = (
    df["team1_previous_10_average_map_score"]
    - df["team2_previous_10_average_map_score"]
)

df[[
    "team1_previous_10_average_map_score",
    "team2_previous_10_average_map_score",
    "map_score_diff"
]].head()

,team1_previous_10_average_map_score,team2_previous_10_average_map_score,map_score_diff
0,13.000000,13.000000,0.0
1,12.333333,12.333333,0.0
2,11.600000,11.600000,0.0
3,11.571429,11.571429,0.0
4,10.666667,10.666667,0.0


In [7]:
numeric_features = [
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "adr_diff",
    "kast_diff",
    "kddiff_diff",
    "map_score_diff",
    "bestOf"
]

categorical_features = [
    "map_name"
]

X = df[numeric_features + categorical_features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (5539, 9)
y shape: (5539,)


,kills_diff,deaths_diff,assists_diff,adr_diff,kast_diff,kddiff_diff,map_score_diff,bestOf,map_name
0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,Overpass
1,0.6,-0.6,-3.2,0.52,1.68,1.2,0.0,3.0,Ancient
2,1.5,-1.6,-0.8,3.11,8.47,3.1,0.0,3.0,Inferno
3,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,Overpass
4,9.0,-9.0,3.0,46.80,48.00,18.0,0.0,3.0,Anubis


In [8]:
##One-hot encode categorical features

X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

print("X shape after one-hot encoding:", X.shape)

X.head()

X shape after one-hot encoding: (5539, 16)


,kills_diff,deaths_diff,assists_diff,adr_diff,kast_diff,kddiff_diff,map_score_diff,bestOf,map_name_Anubis,map_name_Dust2,map_name_Inferno,map_name_Mirage,map_name_Nuke,map_name_Overpass,map_name_Train,map_name_Vertigo
0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,False,False,False,False,False,True,False,False
1,0.6,-0.6,-3.2,0.52,1.68,1.2,0.0,3.0,False,False,False,False,False,False,False,False
2,1.5,-1.6,-0.8,3.11,8.47,3.1,0.0,3.0,False,False,True,False,False,False,False,False
3,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,False,False,False,False,False,True,False,False
4,9.0,-9.0,3.0,46.80,48.00,18.0,0.0,3.0,True,False,False,False,False,False,False,False


In [9]:
# Handle missing values

print("Missing values before filling:")
print(X.isna().sum().sort_values(ascending=False).head(20))

X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

print("\nMissing values after filling:")
print(X.isna().sum().sum())

Missing values before filling:
kills_diff           0
deaths_diff          0
assists_diff         0
adr_diff             0
kast_diff            0
kddiff_diff          0
map_score_diff       0
bestOf               0
map_name_Anubis      0
map_name_Dust2       0
map_name_Inferno     0
map_name_Mirage      0
map_name_Nuke        0
map_name_Overpass    0
map_name_Train       0
map_name_Vertigo     0
dtype: int64

Missing values after filling:
0


In [10]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Train size: (4431, 16)
Test size: (1108, 16)

Training target distribution:
team1_win
1    0.546378
0    0.453622
Name: proportion, dtype: float64

Testing target distribution:
team1_win
1    0.558664
0    0.441336
Name: proportion, dtype: float64


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Random Forest Test Accuracy: 0.5568592057761733

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.39      0.44       489
           1       0.59      0.69      0.63       619

    accuracy                           0.56      1108
   macro avg       0.54      0.54      0.54      1108
weighted avg       0.55      0.56      0.55      1108


Confusion Matrix:
[[190 299]
 [192 427]]


In [13]:
##Check feature importance

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance.head(20)

,feature,importance
4,kast_diff,0.137413
3,adr_diff,0.137125
5,kddiff_diff,0.130813
6,map_score_diff,0.127368
0,kills_diff,0.123402
2,assists_diff,0.123245
1,deaths_diff,0.121073
7,bestOf,0.015958
11,map_name_Mirage,0.012910
10,map_name_Inferno,0.012760


In [14]:
numeric_features_v2 = [
    # Difference features
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "adr_diff",
    "kast_diff",
    "kddiff_diff",
    "map_score_diff",
    
    # Team 1 average features
    "team1_avg_kills",
    "team1_avg_deaths",
    "team1_avg_assists",
    "team1_avg_adr",
    "team1_avg_kast",
    "team1_avg_kddiff",
    
    # Team 2 average features
    "team2_avg_kills",
    "team2_avg_deaths",
    "team2_avg_assists",
    "team2_avg_adr",
    "team2_avg_kast",
    "team2_avg_kddiff",
    
    # Match format
    "bestOf"
]

categorical_features_v2 = [
    "map_name"
]



In [15]:
X_v2 = df[numeric_features_v2 + categorical_features_v2]
y = df[target]

X_v2 = pd.get_dummies(
    X_v2,
    columns=categorical_features_v2,
    drop_first=True
)

X_v2 = X_v2.fillna(X_v2.median(numeric_only=True))
X_v2 = X_v2.fillna(0)

print("X_v2 shape:", X_v2.shape)

X_v2 shape: (5539, 28)


In [16]:
split_index = int(len(df) * 0.8)

X_train_v2 = X_v2.iloc[:split_index]
X_test_v2 = X_v2.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train_v2.shape)
print("Test size:", X_test_v2.shape)

Train size: (4431, 28)
Test size: (1108, 28)


In [17]:
rf_v2 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_v2.fit(X_train_v2, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [18]:
y_pred_v2 = rf_v2.predict(X_test_v2)

print("Random Forest V2 Test Accuracy:", accuracy_score(y_test, y_pred_v2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_v2))

Random Forest V2 Test Accuracy: 0.5342960288808665

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.32      0.38       489
           1       0.57      0.70      0.63       619

    accuracy                           0.53      1108
   macro avg       0.51      0.51      0.50      1108
weighted avg       0.52      0.53      0.52      1108


Confusion Matrix:
[[158 331]
 [185 434]]


### Random Forest using individual player statistics

In [19]:
player_stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

player_features = []

for team in ["team1", "team2"]:
    for player in range(1, 6):
        for stat in player_stats:
            col = f"previous_10_game_{team}_player{player}_average_{stat}"
            player_features.append(col)

print("Number of player-level features:", len(player_features))
print(player_features[:10])

Number of player-level features: 60
['previous_10_game_team1_player1_average_kills', 'previous_10_game_team1_player1_average_deaths', 'previous_10_game_team1_player1_average_assists', 'previous_10_game_team1_player1_average_adr', 'previous_10_game_team1_player1_average_kast', 'previous_10_game_team1_player1_average_kddiff', 'previous_10_game_team1_player2_average_kills', 'previous_10_game_team1_player2_average_deaths', 'previous_10_game_team1_player2_average_assists', 'previous_10_game_team1_player2_average_adr']


In [20]:
X_player = df[player_features]
y = df[target]

print("X_player shape:", X_player.shape)
print("y shape:", y.shape)

X_player.head()

X_player shape: (5539, 60)
y shape: (5539,)


,previous_10_game_team1_player1_average_kills,previous_10_game_team1_player1_average_deaths,previous_10_game_team1_player1_average_assists,previous_10_game_team1_player1_average_adr,previous_10_game_team1_player1_average_kast,previous_10_game_team1_player1_average_kddiff,previous_10_game_team1_player2_average_kills,previous_10_game_team1_player2_average_deaths,previous_10_game_team1_player2_average_assists,previous_10_game_team1_player2_average_adr,...,previous_10_game_team2_player4_average_assists,previous_10_game_team2_player4_average_adr,previous_10_game_team2_player4_average_kast,previous_10_game_team2_player4_average_kddiff,previous_10_game_team2_player5_average_kills,previous_10_game_team2_player5_average_deaths,previous_10_game_team2_player5_average_assists,previous_10_game_team2_player5_average_adr,previous_10_game_team2_player5_average_kast,previous_10_game_team2_player5_average_kddiff
0,16.000000,15.000000,4.000000,75.000000,70.000000,1.000000,16.000000,15.000000,4.000000,75.000000,...,4.000000,75.000000,70.000000,1.000000,16.000000,15.000000,4.000000,75.000000,70.000000,1.000000
1,12.000000,17.000000,4.000000,59.100000,62.500000,-5.000000,13.000000,11.000000,2.000000,52.600000,...,7.000000,80.100000,83.300000,1.000000,13.000000,16.000000,4.000000,49.000000,58.300000,-3.000000
2,13.500000,16.500000,4.500000,66.250000,71.750000,-3.000000,18.500000,11.000000,4.000000,82.550000,...,4.500000,64.050000,75.000000,-1.500000,11.000000,16.500000,5.000000,61.500000,60.100000,-5.500000
3,14.612903,14.645161,4.258065,69.832258,71.535484,-0.032258,14.612903,14.645161,4.258065,69.832258,...,4.258065,69.832258,71.535484,-0.032258,14.612903,14.645161,4.258065,69.832258,71.535484,-0.032258
4,13.000000,5.000000,1.000000,81.300000,86.700000,8.000000,11.000000,6.000000,4.000000,72.100000,...,1.000000,47.300000,26.700000,-11.000000,2.000000,14.000000,0.000000,23.500000,26.700000,-12.000000


In [22]:
split_index = int(len(df) * 0.8)

X_train_player = X_player.iloc[:split_index]
X_test_player = X_player.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train_player.shape)
print("Test size:", X_test_player.shape)

Train size: (4431, 60)
Test size: (1108, 60)


In [23]:
rf_player = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_player.fit(X_train_player, y_train)

y_pred_player = rf_player.predict(X_test_player)

print("Player-level Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_player))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_player))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_player))

Player-level Random Forest Test Accuracy: 0.5442238267148014

Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.30      0.37       489
           1       0.57      0.73      0.64       619

    accuracy                           0.54      1108
   macro avg       0.52      0.52      0.51      1108
weighted avg       0.53      0.54      0.52      1108


Confusion Matrix:
[[149 340]
 [165 454]]


In [24]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_player, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best validation accuracy:", grid_search.best_score_)

Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best validation accuracy: 0.5700743877579585


In [26]:
best_rf_player = grid_search.best_estimator_

y_pred_best_player = best_rf_player.predict(X_test_player)

print("Best Player-level Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_best_player))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_player))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_player))

Best Player-level Random Forest Test Accuracy: 0.5478339350180506

Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.23      0.31       489
           1       0.57      0.80      0.66       619

    accuracy                           0.55      1108
   macro avg       0.52      0.51      0.49      1108
weighted avg       0.53      0.55      0.51      1108


Confusion Matrix:
[[114 375]
 [126 493]]


In [27]:
feature_importance_player = pd.DataFrame({
    "feature": X_train_player.columns,
    "importance": best_rf_player.feature_importances_
})

feature_importance_player = feature_importance_player.sort_values(
    by="importance",
    ascending=False
)

feature_importance_player.head(20)

,feature,importance
16,previous_10_game_team1_player3_average_kast,0.028867
10,previous_10_game_team1_player2_average_kast,0.023972
22,previous_10_game_team1_player4_average_kast,0.023778
46,previous_10_game_team2_player3_average_kast,0.021776
40,previous_10_game_team2_player2_average_kast,0.020867
52,previous_10_game_team2_player4_average_kast,0.020491
23,previous_10_game_team1_player4_average_kddiff,0.020335
34,previous_10_game_team2_player1_average_kast,0.020334
27,previous_10_game_team1_player5_average_adr,0.020041
51,previous_10_game_team2_player4_average_adr,0.019995
